### cost_of_living_us.csv cleaning script:


In [ ]:
import pandas as pd
import os

# Load --------------------------------------------------------------------------------
df = pd.read_csv(r"c:\Users\ralph\projects\thrivot\data\raw datasets\cost_of_living_us.csv")

print(f"Shape before cleaning: {df.shape}")

#  1. Drop duplicate rows -------------------------------------------------------
df = df.drop_duplicates()

# -- 2. Drop rows with null county, state, or areaname ------------------------
df = df.dropna(subset=["county", "state", "areaname"])

# -- 3. Standardize state to uppercase 2-letter abbreviations -----------------
df["state"] = df["state"].astype(str).str.strip().str.upper()

# -- 4. Standardize county column ----------------------------------------------
#    lowercase → strip whitespace → remove trailing " county"
df["county"] = (
    df["county"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s*county\s*$", "", regex=True)
    .str.strip()
)

# -- 5. Create county_state_key ------------------------------------------------
df["county_state_key"] = (
    df["county"].str.replace(r"\s+", "_", regex=True)
    + "_"
    + df["state"].str.lower()
)

# -- 6. Keep only rows where total family size (parents + children) is 1–4 -----
#    family_member_count is formatted as e.g. "1p0c", "2p3c"
#    Parse parents and children from the string and sum them.
def parse_family_size(val):
    val = str(val).strip().lower()
    try:
        parents = int(val.split("p")[0])
        children = int(val.split("p")[1].replace("c", ""))
        return parents + children
    except (IndexError, ValueError):
        return None

df["family_size"] = df["family_member_count"].apply(parse_family_size)
df = df[df["family_size"].isin([1, 2, 3, 4])].drop(columns=["family_size"])

# -- 7. Convert cost columns to numeric ---------------------------------------
cost_cols = [
    "housing_cost",
    "food_cost",
    "transportation_cost",
    "healthcare_cost",
    "other_necessities_cost",
    "childcare_cost",
    "taxes",
    "total_cost",
    "median_family_income",
]
for col in cost_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -- 8. Drop rows where total_cost is null or zero ----------------------------
df = df[df["total_cost"].notna() & (df["total_cost"] != 0)]

print(f"Shape after cleaning:  {df.shape}")

# -- 9. Save -------------------------------------------------------------------
output_path = r"c:\Users\ralph\projects\thrivot\data\cleaned_data\cost_of_living_us_cleaned.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


Shape before cleaning: (31430, 15)
Shape after cleaning:  (22001, 16)
Saved cleaned dataset to: c:\Users\ralph\projects\thrivot\data\cleaned_data\cost_of_living_us_cleaned.csv


### zillow_zori_rent.csv script

In [2]:
import pandas as pd
import os

# -- Load ------------------------------------------------------------------
df = pd.read_csv(r"c:\Users\ralph\projects\thrivot\data\raw datasets\zillow_zori_rent.csv")

print(f"Shape before cleaning: {df.shape}")

# -- 1. Drop duplicate rows ------------------------------------------------
df = df.drop_duplicates()

# -- 2. Filter to city-level regions, drop rows missing key fields ---------
df = df[df["RegionType"] == "city"]
df = df.dropna(subset=["RegionID", "State", "CountyName"])

# -- 3. Standardize State to uppercase 2-letter abbreviations --------------
df["State"] = df["State"].astype(str).str.strip().str.upper()

# -- 4. Standardize CountyName - used later to join with cost-of-living ----
#    lowercase -> strip whitespace -> remove trailing " county"
df["county"] = (
    df["CountyName"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s*county\s*$", "", regex=True)
    .str.strip()
)

# -- 5. Create county_state_key --------------------------------------------
df["county_state_key"] = (
    df["county"].str.replace(r"\s+", "_", regex=True)
    + "_"
    + df["State"].str.lower()
)

# -- 6. Melt from wide to long - date columns become rows ------------------
#    Identify date columns (YYYY-MM-DD format), keep identifier columns
id_cols = ["RegionID", "RegionName", "State", "Metro", "county", "county_state_key"]
drop_cols = {"SizeRank", "RegionType", "StateName", "CountyName"}
date_cols = [c for c in df.columns if c not in id_cols and c not in drop_cols]

df = df.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="date",
    value_name="zori",
)

# -- 7. Parse date column to datetime --------------------------------------
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["date"].notna()]

# -- 8. Drop rows where zori is null or zero -------------------------------
df["zori"] = pd.to_numeric(df["zori"], errors="coerce")
df = df[df["zori"].notna() & (df["zori"] != 0)]

print(f"Shape after cleaning:  {df.shape}")

# -- 9. Save ---------------------------------------------------------------
output_path = r"c:\Users\ralph\projects\thrivot\data\cleaned_data\zillow_zori_rent_cleaned.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


Shape before cleaning: (4430, 144)
Shape after cleaning:  (206746, 8)
Saved cleaned dataset to: c:\Users\ralph\projects\thrivot\data\cleaned_data\zillow_zori_rent_cleaned.csv


### NRI_Table_Counties.csv cleaning script (FEMA National Risk Index)


In [3]:
import pandas as pd
import os

# -- Load ------------------------------------------------------------------
df = pd.read_csv(r"c:\Users\ralph\projects\thrivot\data\raw datasets\weather data\NRI_Table_Counties.csv", encoding="utf-8-sig")

print(f"Shape before cleaning: {df.shape}")

# -- 1. Drop duplicate rows ------------------------------------------------
df = df.drop_duplicates()

# -- 2. Keep only columns needed for the app -------------------------------
#    Identifiers + the 4 hazard risk scores shown in the app UI
keep_cols = [
    "STATEABBRV",
    "COUNTY",
    "STCOFIPS",
    "CFLD_RISKS",   # coastal flood risk score
    "IFLD_RISKS",   # inland flood risk score
    "HRCN_RISKS",   # hurricane risk score
    "WFIR_RISKS",   # wildfire risk score
    "TRND_RISKS",   # tornado risk score
]
df = df[keep_cols]

# -- 3. Standardize state abbreviation to uppercase ------------------------
df["STATEABBRV"] = df["STATEABBRV"].astype(str).str.strip().str.upper()

# -- 4. Standardize county column ------------------------------------------
#    lowercase -> strip whitespace -> remove trailing " county"
df["county"] = (
    df["COUNTY"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s*county\s*$", "", regex=True)
    .str.strip()
)
df = df.drop(columns=["COUNTY"])

# -- 5. Create county_state_key --------------------------------------------
df["county_state_key"] = (
    df["county"].str.replace(r"\s+", "_", regex=True)
    + "_"
    + df["STATEABBRV"].str.lower()
)

# -- 6. Combine coastal and inland flood into a single flood risk score ----
#    Use the higher of the two; if both are null, result is null
df["CFLD_RISKS"] = pd.to_numeric(df["CFLD_RISKS"], errors="coerce")
df["IFLD_RISKS"] = pd.to_numeric(df["IFLD_RISKS"], errors="coerce")
df["flood_risk_score"] = df[["CFLD_RISKS", "IFLD_RISKS"]].max(axis=1)
df = df.drop(columns=["CFLD_RISKS", "IFLD_RISKS"])

# -- 7. Convert remaining risk score columns to numeric --------------------
risk_cols = ["HRCN_RISKS", "WFIR_RISKS", "TRND_RISKS"]
for col in risk_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -- 8. Rename risk columns to readable names ------------------------------
df = df.rename(columns={
    "STATEABBRV": "state",
    "STCOFIPS":   "county_fips",
    "HRCN_RISKS": "hurricane_risk_score",
    "WFIR_RISKS": "wildfire_risk_score",
    "TRND_RISKS": "tornado_risk_score",
})

# -- 9. Drop rows missing county_state_key or all four risk scores ---------
df = df.dropna(subset=["county_state_key"])
df = df[df[["flood_risk_score", "hurricane_risk_score", "wildfire_risk_score", "tornado_risk_score"]].notna().any(axis=1)]

print(f"Shape after cleaning:  {df.shape}")

# -- 10. Save --------------------------------------------------------------
output_path = r"c:\Users\ralph\projects\thrivot\data\cleaned_data\nri_weather_risk_cleaned.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


Shape before cleaning: (3232, 465)
Shape after cleaning:  (3144, 8)
Saved cleaned dataset to: c:\Users\ralph\projects\thrivot\data\cleaned_data\nri_weather_risk_cleaned.csv


### crime data cleaning script (FBI UCR - all population bands)


In [4]:
import pandas as pd
import os
import re

# -- State name -> abbreviation lookup -------------------------------------
STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

# -- Load all 4 crime files and normalize columns --------------------------
#    Files split cities by population band: 40-60k, 60-100k, 100-250k, 250k+
#    250k+ file uses different column names; rename to match the others

base = r"c:\Users\ralph\projects\thrivot\data\raw datasets\crime data"

df_small  = pd.read_csv(os.path.join(base, "crime_40_60.csv"))
df_mid1   = pd.read_csv(os.path.join(base, "crime_60_100.csv"))
df_mid2   = pd.read_csv(os.path.join(base, "crime_100_250.csv"))
df_large  = pd.read_csv(os.path.join(base, "crime_250_plus.csv"))

#    Rename 250k+ columns to match the shared schema
df_large = df_large.rename(columns={
    "tot_violent_crime": "violent_crime",
    "tot_prop_crim":     "prop_crime",
})

#    Stack all bands into one dataframe
shared_cols = ["states", "cities", "population", "violent_crime", "prop_crime"]
df = pd.concat(
    [df_small[shared_cols], df_mid1[shared_cols], df_mid2[shared_cols], df_large[shared_cols]],
    ignore_index=True,
)

print(f"Shape after concat: {df.shape}")

# -- 1. Drop duplicate rows ------------------------------------------------
df = df.drop_duplicates()

# -- 2. Clean state column -------------------------------------------------
#    Strip BOM chars and extra whitespace, then map full name -> abbreviation
df["states"] = df["states"].astype(str).str.replace("\xa0", "", regex=False).str.strip()
df["state"] = df["states"].map(STATE_ABBREV)
df = df.dropna(subset=["state"])
df = df.drop(columns=["states"])

# -- 3. Clean city column --------------------------------------------------
#    40-60k entries include county: "Abington Township, Montgomery County"
#    -> take only the part before the first comma
#    250k+ entries have trailing digits: "Mobile3" -> "Mobile"
df["city"] = (
    df["cities"]
    .astype(str)
    .str.split(",").str[0]          # strip county suffix if present
    .str.replace(r"\d+$", "", regex=True)  # strip trailing digits
    .str.strip()
    .str.lower()
)
df = df.drop(columns=["cities"])

# -- 4. Drop rows missing city or with empty city name ---------------------
df = df[df["city"].notna() & (df["city"] != "")]

# -- 5. Create city_state_key ----------------------------------------------
df["city_state_key"] = (
    df["city"].str.replace(r"\s+", "_", regex=True)
    + "_"
    + df["state"].str.lower()
)

# -- 6. Convert numeric columns - remove commas, cast to float -------------
for col in ["population", "violent_crime", "prop_crime"]:
    df[col] = (
        df[col].astype(str)
        .str.replace(",", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

# -- 7. Drop rows where both crime rates are null --------------------------
df = df[df[["violent_crime", "prop_crime"]].notna().any(axis=1)]

# -- 8. Rename to clear output column names --------------------------------
df = df.rename(columns={
    "violent_crime": "violent_crime_rate",
    "prop_crime":    "property_crime_rate",
})

print(f"Shape after cleaning:  {df.shape}")
print(df[["city", "state", "city_state_key", "violent_crime_rate", "property_crime_rate"]].head())

# -- 9. Save ---------------------------------------------------------------
output_path = r"c:\Users\ralph\projects\thrivot\data\cleaned_data\crime_cleaned.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


Shape after concat: (975, 5)
Shape after cleaning:  (972, 6)
                city state        city_state_key  violent_crime_rate  \
0  abington township    PA  abington_township_pa               197.4   
1             albany    OR             albany_or                86.1   
2         alexandria    LA         alexandria_la              1682.2   
3        aliso viejo    CA        aliso_viejo_ca                87.8   
4  altamonte springs    FL  altamonte_springs_fl               335.7   

   property_crime_rate  
0               1979.1  
1               3092.9  
2               7492.4  
3                847.0  
4               3057.0  
Saved cleaned dataset to: c:\Users\ralph\projects\thrivot\data\cleaned_data\crime_cleaned.csv


### uscities_latlon.csv cleaning script (city coordinates for map rendering)


In [5]:
import pandas as pd
import os

# -- Load ------------------------------------------------------------------
df = pd.read_csv(r"c:\Users\ralph\projects\thrivot\data\raw datasets\uscities_latlon.csv")

print(f"Shape before cleaning: {df.shape}")

# -- 1. Drop duplicate rows ------------------------------------------------
df = df.drop_duplicates()

# -- 2. Keep only columns needed for the app -------------------------------
#    city name, state, county (for joins), lat/lng (for map pins)
keep_cols = ["city_ascii", "state_id", "county_name", "lat", "lng"]
df = df[keep_cols]

# -- 3. Standardize state to uppercase 2-letter abbreviations --------------
df["state_id"] = df["state_id"].astype(str).str.strip().str.upper()

# -- 4. Standardize city name ----------------------------------------------
#    Use city_ascii (no accents) for clean key generation
df["city"] = df["city_ascii"].astype(str).str.strip().str.lower()
df = df.drop(columns=["city_ascii"])

# -- 5. Standardize county name --------------------------------------------
#    lowercase -> strip whitespace -> remove trailing " county"
#    Kept to join this dataset with county-level data later
df["county"] = (
    df["county_name"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s*county\s*$", "", regex=True)
    .str.strip()
)
df = df.drop(columns=["county_name"])

# -- 6. Create city_state_key and county_state_key -------------------------
df["city_state_key"] = (
    df["city"].str.replace(r"\s+", "_", regex=True)
    + "_"
    + df["state_id"].str.lower()
)
df["county_state_key"] = (
    df["county"].str.replace(r"\s+", "_", regex=True)
    + "_"
    + df["state_id"].str.lower()
)

# -- 7. Convert lat/lng to numeric -----------------------------------------
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lng"] = pd.to_numeric(df["lng"], errors="coerce")

# -- 8. Drop rows missing lat, lng, or city --------------------------------
df = df.dropna(subset=["lat", "lng", "city"])

# -- 9. Rename state column ------------------------------------------------
df = df.rename(columns={"state_id": "state"})

print(f"Shape after cleaning:  {df.shape}")

# -- 10. Save --------------------------------------------------------------
output_path = r"c:\Users\ralph\projects\thrivot\data\cleaned_data\uscities_latlon_cleaned.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")


Shape before cleaning: (28338, 17)
Shape after cleaning:  (28338, 7)
Saved cleaned dataset to: c:\Users\ralph\projects\thrivot\data\cleaned_data\uscities_latlon_cleaned.csv
